In [ ]:

import os
import sys
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
OUT = "output"
os.makedirs(OUT, exist_ok=True)
log = []  # cleaning log used in the report


def find_file(name="city_day.csv"):
    for p in (name, os.path.join("data", name)):
        if os.path.exists(p):
            return p
    sys.exit(f"{name} not found. Download it from the Kaggle link and place it next to this script.")


def save(fig_name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, fig_name), dpi=150)
    plt.close()


# ------------------------------------------------------------------ 1. LOAD
file_path = find_file()

try:
    # Since the file starts with PK\x03\x04, it is a ZIP archive.
    # We will read 'city_day.csv' from inside the zip file.
    with zipfile.ZipFile(file_path, 'r') as z:
        # Find if city_day.csv or any csv exists inside the ZIP
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
        if csv_files:
            csv_to_extract = csv_files[0]
            print(f"Extracting and loading {csv_to_extract} from ZIP archive...")
            with z.open(csv_to_extract) as f:
                df = pd.read_csv(f, encoding="utf-8")
        else:
            raise ValueError("No CSV file found inside the ZIP archive.")
except (zipfile.BadZipFile, KeyError, ValueError) as e:
    print("File is not a valid ZIP or extraction failed, trying direct CSV read:", e)
    try:
        df = pd.read_csv(file_path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin-1")

raw_shape = df.shape
print("Raw shape:", raw_shape)
print(df.dtypes, "\n")
log.append(f"Raw data: {raw_shape[0]:,} rows x {raw_shape[1]} columns.")

POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
              "Benzene", "Toluene", "Xylene"]

# ------------------------------------------------------------------ 2. DATA TYPES
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["City"] = df["City"].astype("category")
df["AQI_Bucket"] = df["AQI_Bucket"].astype("category")
for c in POLLUTANTS + ["AQI"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
log.append("Converted Date to datetime, City/AQI_Bucket to category, pollutants to numeric.")

# ------------------------------------------------------------------ 3. DUPLICATES
dups = df.duplicated().sum()
dups_key = df.duplicated(subset=["City", "Date"]).sum()
df = df.drop_duplicates().drop_duplicates(subset=["City", "Date"], keep="first")
log.append(f"Removed {dups} exact duplicates and {dups_key} duplicate City+Date records.")

# ------------------------------------------------------------------ 4. MISSING VALUES
miss_before = (df.isna().mean() * 100).sort_values(ascending=False)

# Viz 1: missing values before cleaning
plt.figure(figsize=(9, 5))
sns.barplot(x=miss_before.values, y=miss_before.index, color="#d9534f")
plt.xlabel("% missing"); plt.title("Missing values per column (before cleaning)")
save("1_missing_values.png")

# a) drop rows with no AQI (target variable)
n0 = len(df)
df = df.dropna(subset=["AQI"])
log.append(f"Dropped {n0 - len(df):,} rows where AQI (target) was missing.")

# b) drop columns with > 60% missing
heavy = [c for c in POLLUTANTS if df[c].isna().mean() > 0.60]
df = df.drop(columns=heavy)
POLLUTANTS = [c for c in POLLUTANTS if c not in heavy]
log.append(f"Dropped columns with >60% missing values: {heavy if heavy else 'none'}.")

# c) impute remaining with city-wise median, fallback global median
for c in POLLUTANTS:
    df[c] = df.groupby("City", observed=True)[c].transform(lambda s: s.fillna(s.median()))
    df[c] = df[c].fillna(df[c].median())
log.append("Imputed remaining missing pollutant values with city-wise median (global median fallback).")

# d) rebuild AQI_Bucket from AQI using the official CPCB ranges
def bucket(a):
    if a <= 50: return "Good"
    if a <= 100: return "Satisfactory"
    if a <= 200: return "Moderate"
    if a <= 300: return "Poor"
    if a <= 400: return "Very Poor"
    return "Severe"

df["AQI_Bucket"] = df["AQI"].apply(bucket)
ORDER = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
df["AQI_Bucket"] = pd.Categorical(df["AQI_Bucket"], categories=ORDER, ordered=True)

# ------------------------------------------------------------------ 5. OUTLIERS
df_before = df.copy()
num_cols = POLLUTANTS + ["AQI"]

neg = (df[num_cols] < 0).sum().sum()
for c in num_cols:                       # negative concentrations are impossible
    df.loc[df[c] < 0, c] = np.nan
    df[c] = df[c].fillna(df[c].median())
log.append(f"Fixed {neg} impossible negative readings (replaced by median).")

outlier_counts = {}
for c in num_cols:                       # IQR detection, winsorize at 1st/99th pct
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_counts[c] = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    lo, hi = df[c].quantile([0.01, 0.99])
    df[c] = df[c].clip(lo, hi)
log.append("Outliers detected with 1.5xIQR rule; extreme values capped at 1st/99th percentile "
           "(high pollution days are real, so rows were kept, not deleted).")
log.append("Outlier counts (IQR): " + ", ".join(f"{k}={v}" for k, v in outlier_counts.items()))

# Viz 2: outliers before vs after
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
sns.boxplot(data=df_before[["PM2.5", "PM10", "NO2", "SO2"]], ax=ax[0], palette="Reds")
ax[0].set_title("Before outlier handling")
sns.boxplot(data=df[["PM2.5", "PM10", "NO2", "SO2"]], ax=ax[1], palette="Greens")
ax[1].set_title("After outlier handling")
save("2_outliers_before_after.png")

# feature engineering
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df = df.sort_values(["City", "Date"]).reset_index(drop=True)
df.to_csv(os.path.join(OUT, "city_day_cleaned.csv"), index=False)
log.append(f"Final cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns.")

# ------------------------------------------------------------------ 6. VISUALIZATIONS
# Viz 3: most polluted cities
city_aqi = df.groupby("City", observed=True)["AQI"].mean().sort_values(ascending=False)
plt.figure(figsize=(9, 5))
sns.barplot(x=city_aqi.head(10).values, y=city_aqi.head(10).index.astype(str), palette="OrRd_r")
plt.xlabel("Average AQI"); plt.title("Top 10 most polluted cities")
save("3_top_polluted_cities.png")

# Viz 4: yearly trend
yearly = df.groupby("Year")["AQI"].mean()
plt.figure(figsize=(9, 5))
plt.plot(yearly.index, yearly.values, marker="o", color="#c0392b")
plt.xlabel("Year"); plt.ylabel("Average AQI"); plt.title("National average AQI by year")
save("4_yearly_trend.png")

# Viz 5: monthly seasonality
monthly = df.groupby("Month")["AQI"].mean()
plt.figure(figsize=(9, 5))
sns.barplot(x=monthly.index, y=monthly.values, palette="coolwarm")
plt.xlabel("Month"); plt.ylabel("Average AQI"); plt.title("Seasonality: average AQI by month")
save("5_monthly_seasonality.png")

# Viz 6: correlation heatmap
plt.figure(figsize=(9, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", square=True)
plt.title("Correlation between pollutants and AQI")
save("6_correlation_heatmap.png")

# Viz 7: AQI bucket distribution
plt.figure(figsize=(8, 5))
cnt = df["AQI_Bucket"].value_counts().reindex(ORDER)
sns.barplot(x=cnt.index.astype(str), y=cnt.values,
            palette=["#2ecc71", "#a3d977", "#f1c40f", "#e67e22", "#e74c3c", "#7b241c"])
plt.ylabel("Number of days"); plt.title("Distribution of AQI categories")
save("7_aqi_bucket_distribution.png")

# Viz 8: lockdown effect (Mar-Jun 2019 vs 2020)
sub = df[df["Month"].between(3, 6) & df["Year"].isin([2019, 2020])]
if sub["Year"].nunique() == 2:
    plt.figure(figsize=(9, 5))
    top5 = city_aqi.head(5).index
    sns.barplot(data=sub[sub["City"].isin(top5)], x="City", y="AQI", hue="Year", errorbar=None)
    plt.title("COVID lockdown effect: AQI Mar-Jun 2019 vs 2020 (top 5 polluted cities)")
    save("8_lockdown_effect.png")

# ------------------------------------------------------------------ 7. INSIGHTS REPORT
worst, best = city_aqi.index[0], city_aqi.index[-1]
worst_m, best_m = monthly.idxmax(), monthly.idxmin()
top_corr = corr["AQI"].drop("AQI").sort_values(ascending=False)
pct_bad = (df["AQI_Bucket"].isin(["Poor", "Very Poor", "Severe"]).mean()) * 100
y_hi, y_lo = yearly.idxmax(), yearly.idxmin()
months = {1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June", 7: "July",
          8: "August", 9: "September", 10: "October", 11: "November", 12: "December"}

lock = ""
if sub["Year"].nunique() == 2:
    a = sub[sub.Year == 2019]["AQI"].mean(); b = sub[sub.Year == 2020]["AQI"].mean()
    lock = (f"- **Lockdown effect:** Mean AQI during Mar-Jun fell from {a:.0f} (2019) to {b:.0f} "
            f"(2020), a change of {(b - a) / a * 100:.1f}%, showing the impact of reduced traffic/industry.\n")

report = f"""# Air Quality in India - Insights Report
**Student:** AKSHAYSINH CHAUHAN (IU2441230574) | **Team:** TEAM ANSH
**Dataset:** Air Quality Data in India (Kaggle, `city_day.csv`)

## 1. Data Cleaning Summary
""" + "\n".join(f"- {l}" for l in log) + f"""

## 2. Key Insights
- **Most polluted city:** {worst} (avg AQI {city_aqi.iloc[0]:.0f}); cleanest in dataset: {best} (avg AQI {city_aqi.iloc[-1]:.0f}).
- **Trend:** Average AQI was highest in {y_hi} ({yearly.max():.0f}) and lowest in {y_lo} ({yearly.min():.0f}).
- **Seasonality:** Pollution peaks in {months[worst_m]} (AQI {monthly.max():.0f}) and is lowest in {months[best_m]} (AQI {monthly.min():.0f}); winter is typically worse because of crop burning, cold air trapping pollutants and low wind, while monsoon rain washes particles out.
- **Main drivers:** Pollutants most correlated with AQI are {top_corr.index[0]} (r={top_corr.iloc[0]:.2f}) and {top_corr.index[1]} (r={top_corr.iloc[1]:.2f}); particulate matter is the dominant factor.
- **Health risk:** {pct_bad:.1f}% of recorded city-days fell in the Poor, Very Poor or Severe category.
{lock}
## 3. Visualizations
1. Missing values per column (`1_missing_values.png`)
2. Outliers before vs after (`2_outliers_before_after.png`)
3. Top 10 polluted cities (`3_top_polluted_cities.png`)
4. Yearly AQI trend (`4_yearly_trend.png`)
5. Monthly seasonality (`5_monthly_seasonality.png`)
6. Correlation heatmap (`6_correlation_heatmap.png`)
7. AQI category distribution (`7_aqi_bucket_distribution.png`)
8. Lockdown effect 2019 vs 2020 (`8_lockdown_effect.png`)

## 4. Conclusion
Air pollution in India is concentrated in a few northern/industrial cities and is strongly seasonal.
Controlling particulate emissions (vehicles, construction dust, biomass burning) would give the
largest improvement in AQI. The lockdown shows that emission cuts quickly improve air quality.
"""
with open(os.path.join(OUT, "insights_report.md"), "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print(f"\nDone. Check the '{OUT}' folder.")

Extracting and loading city_day.csv from ZIP archive...
Raw shape: (29531, 16)
City           object
Date           object
PM2.5         float64
PM10          float64
NO            float64
NO2           float64
NOx           float64
NH3           float64
CO            float64
SO2           float64
O3            float64
Benzene       float64
Toluene       float64
Xylene        float64
AQI           float64
AQI_Bucket     object
dtype: object 



/tmp/ipykernel_744/3383554860.py:165: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=city_aqi.head(10).values, y=city_aqi.head(10).index.astype(str), palette="OrRd_r")
/tmp/ipykernel_744/3383554860.py:179: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=monthly.index, y=monthly.values, palette="coolwarm")
/tmp/ipykernel_744/3383554860.py:193: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=cnt.index.astype(str), y=cnt.values,


# Air Quality in India - Insights Report
**Student:** AKSHAYSINH CHAUHAN (IU2441230574) | **Team:** TEAM ANSH
**Dataset:** Air Quality Data in India (Kaggle, `city_day.csv`)

## 1. Data Cleaning Summary
- Raw data: 29,531 rows x 16 columns.
- Converted Date to datetime, City/AQI_Bucket to category, pollutants to numeric.
- Removed 0 exact duplicates and 0 duplicate City+Date records.
- Dropped 4,681 rows where AQI (target) was missing.
- Dropped columns with >60% missing values: ['Xylene'].
- Imputed remaining missing pollutant values with city-wise median (global median fallback).
- Fixed 0 impossible negative readings (replaced by median).
- Outliers detected with 1.5xIQR rule; extreme values capped at 1st/99th percentile (high pollution days are real, so rows were kept, not deleted).
- Outlier counts (IQR): PM2.5=2004, PM10=1534, NO=2351, NO2=1107, NOx=1937, NH3=1149, CO=2303, SO2=2435, O3=739, Benzene=1786, Toluene=3120, AQI=1358
- Final cleaned data: 24,850 rows x 17 columns.

## 